In [ ]:
# This file contains code written by AI.

import pandas as pd
import numpy as np
from pathlib import Path

# Load the success rate CSV files
success_rate_dir = Path("../../../tower_of_london_human_data/problems_ordered")

df_healthy = pd.read_csv(success_rate_dir / "success_rate_Healthy.csv")
df_mci = pd.read_csv(success_rate_dir / "success_rate_MCI.csv")
df_pd = pd.read_csv(success_rate_dir / "success_rate_PD.csv")
df_stroke = pd.read_csv(success_rate_dir / "success_rate_Stroke.csv")

print("Loaded success rate files:")
print(f"Healthy: {len(df_healthy)} problems")
print(f"MCI: {len(df_mci)} problems")
print(f"PD: {len(df_pd)} problems")
print(f"Stroke: {len(df_stroke)} problems")

Loaded success rate files:
Healthy: 24 problems
MCI: 24 problems
PD: 24 problems
Stroke: 24 problems


In [4]:
# Merge all success rates and calculate average
merged_df = df_healthy.copy()
merged_df.columns = ['TOL-ID', 'success_rate_healthy']

merged_df = merged_df.merge(df_mci[['TOL-ID', 'success_rate']], on='TOL-ID')
merged_df.rename(columns={'success_rate': 'success_rate_mci'}, inplace=True)

merged_df = merged_df.merge(df_pd[['TOL-ID', 'success_rate']], on='TOL-ID')
merged_df.rename(columns={'success_rate': 'success_rate_pd'}, inplace=True)

merged_df = merged_df.merge(df_stroke[['TOL-ID', 'success_rate']], on='TOL-ID')
merged_df.rename(columns={'success_rate': 'success_rate_stroke'}, inplace=True)

# Calculate average success rate across all groups
merged_df['avg_success_rate'] = merged_df[['success_rate_healthy', 'success_rate_mci', 'success_rate_pd', 'success_rate_stroke']].mean(axis=1)

# Sort by average success rate to find the 12 hardest problems
merged_df_sorted = merged_df.sort_values('avg_success_rate')

print("\nAll problems sorted by average success rate (hardest first):")
print(merged_df_sorted[['TOL-ID', 'avg_success_rate']])

# Keep the 12 hardest problems
hardest_12 = merged_df_sorted.head(12)['TOL-ID'].str.replace('TOL-ID-', '').astype(int).tolist()
hardest_12.sort()

print(f"\n12 Hardest problems (by average success rate): {hardest_12}")


All problems sorted by average success rate (hardest first):
       TOL-ID  avg_success_rate
1   TOL-ID-20          0.456524
2   TOL-ID-23          0.474568
0   TOL-ID-12          0.489501
3   TOL-ID-24          0.533846
11  TOL-ID-17          0.554510
5   TOL-ID-18          0.558177
9   TOL-ID-21          0.561436
7   TOL-ID-22          0.564852
6   TOL-ID-19          0.573491
4   TOL-ID-10          0.599353
8   TOL-ID-11          0.691568
10  TOL-ID-15          0.716015
12  TOL-ID-16          0.740137
15   TOL-ID-4          0.765965
14  TOL-ID-13          0.856130
16  TOL-ID-14          0.857537
13   TOL-ID-9          0.886912
17   TOL-ID-7          0.940731
20   TOL-ID-2          0.945625
19   TOL-ID-1          0.946995
18   TOL-ID-3          0.947936
21   TOL-ID-8          0.958554
23   TOL-ID-5          0.968627
22   TOL-ID-6          0.976244

12 Hardest problems (by average success rate): [10, 11, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24]


In [5]:
# Create 20 random 6/6 splits from the 12 hardest problems
np.random.seed(42)

splits = []
for i in range(20):
    # Shuffle the 12 problems
    shuffled = hardest_12.copy()
    np.random.shuffle(shuffled)
    
    # Split into train (first 6) and validation (last 6)
    train = sorted(shuffled[:6])
    validation = sorted(shuffled[6:])
    
    # Convert to space-separated strings with zero-padding
    train_str = ' '.join([f'{p:02d}' for p in train])
    val_str = ' '.join([f'{p:02d}' for p in validation])
    
    splits.append({
        'train_problems': train_str,
        'validation_problems': val_str
    })

# Create DataFrame
splits_df = pd.DataFrame(splits)

print(f"\nCreated {len(splits_df)} random 6/6 splits:")
print(splits_df)


Created 20 random 6/6 splits:
       train_problems validation_problems
0   10 12 18 21 22 23   11 15 17 19 20 24
1   10 15 19 21 22 23   11 12 17 18 20 24
2   11 15 17 19 20 23   10 12 18 21 22 24
3   10 11 18 20 22 23   12 15 17 19 21 24
4   10 12 18 19 20 24   11 15 17 21 22 23
5   17 18 21 22 23 24   10 11 12 15 19 20
6   10 12 17 19 20 24   11 15 18 21 22 23
7   10 11 12 15 18 23   17 19 20 21 22 24
8   11 12 15 17 18 20   10 19 21 22 23 24
9   11 15 17 18 19 22   10 12 20 21 23 24
10  10 12 17 18 22 23   11 15 19 20 21 24
11  11 15 20 21 22 23   10 12 17 18 19 24
12  10 11 15 17 18 22   12 19 20 21 23 24
13  10 11 17 18 21 22   12 15 19 20 23 24
14  10 11 12 19 23 24   15 17 18 20 21 22
15  11 12 18 19 20 23   10 15 17 21 22 24
16  15 18 19 20 22 24   10 11 12 17 21 23
17  11 12 17 19 20 23   10 15 18 21 22 24
18  12 17 18 19 22 24   10 11 15 20 21 23
19  10 15 21 22 23 24   11 12 17 18 19 20


In [6]:
# Load mean_extra_moves_on_success CSV files (mean extra moves on success)
import pandas as pd
import numpy as np
from pathlib import Path

mm_dir = Path("../../../tower_of_london_human_data/problems_ordered")

df_mm_healthy = pd.read_csv(mm_dir / "mean_extra_moves_on_success_Healthy.csv")
df_mm_mci = pd.read_csv(mm_dir / "mean_extra_moves_on_success_MCI.csv")
df_mm_pd = pd.read_csv(mm_dir / "mean_extra_moves_on_success_PD.csv")
df_mm_stroke = pd.read_csv(mm_dir / "mean_extra_moves_on_success_Stroke.csv")

print("Loaded mean-extra-moves files:")
print(f"Healthy: {len(df_mm_healthy)} rows, MCI: {len(df_mm_mci)} rows, PD: {len(df_mm_pd)} rows, Stroke: {len(df_mm_stroke)} rows")

Loaded mean-extra-moves files:
Healthy: 24 rows, MCI: 24 rows, PD: 24 rows, Stroke: 24 rows


In [7]:
# Merge mean-extra-moves from groups and compute average mean_extra_moves_on_success
mm_merged = df_mm_healthy.copy()
mm_merged.columns = ["TOL-ID", "mean_extra_moves_healthy"]

mm_merged = mm_merged.merge(df_mm_mci[['TOL-ID', 'mean_extra_moves_on_success']], on='TOL-ID')
mm_merged.rename(columns={'mean_extra_moves_on_success': 'mean_extra_moves_mci'}, inplace=True)

mm_merged = mm_merged.merge(df_mm_pd[['TOL-ID', 'mean_extra_moves_on_success']], on='TOL-ID')
mm_merged.rename(columns={'mean_extra_moves_on_success': 'mean_extra_moves_pd'}, inplace=True)

mm_merged = mm_merged.merge(df_mm_stroke[['TOL-ID', 'mean_extra_moves_on_success']], on='TOL-ID')
mm_merged.rename(columns={'mean_extra_moves_on_success': 'mean_extra_moves_stroke'}, inplace=True)

mm_merged['mean_extra_moves_on_success_avg'] = mm_merged[['mean_extra_moves_healthy', 'mean_extra_moves_mci', 'mean_extra_moves_pd', 'mean_extra_moves_stroke']].mean(axis=1)

# Higher average extra moves => harder problem
mm_sorted = mm_merged.sort_values('mean_extra_moves_on_success_avg', ascending=False)

print("\nProblems sorted by mean extra moves on success (hardest first):")
print(mm_sorted[["TOL-ID", "mean_extra_moves_on_success_avg"]].to_string(index=False))

# Select 12 hardest problems by this metric
hardest_12_moves = mm_sorted.head(12)['TOL-ID'].str.replace('TOL-ID-', '').astype(int).tolist()
hardest_12_moves.sort()
print(f"\n12 hardest problems by mean_extra_moves_on_success: {hardest_12_moves}")


Problems sorted by mean extra moves on success (hardest first):
   TOL-ID  mean_extra_moves_on_success_avg
TOL-ID-21                         2.109595
TOL-ID-20                         2.055556
TOL-ID-23                         2.031434
TOL-ID-12                         2.002321
TOL-ID-17                         1.940430
TOL-ID-22                         1.792199
TOL-ID-18                         1.659600
TOL-ID-19                         1.453574
TOL-ID-10                         1.368017
TOL-ID-24                         1.297498
TOL-ID-11                         1.278987
TOL-ID-16                         0.838869
 TOL-ID-9                         0.804167
 TOL-ID-4                         0.625330
TOL-ID-15                         0.565153
 TOL-ID-3                         0.388456
TOL-ID-14                         0.354152
 TOL-ID-5                         0.317395
 TOL-ID-6                         0.274479
 TOL-ID-8                         0.248020
TOL-ID-13                       

In [8]:
# Create 20 random 6/6 splits from the 12 hardest by mean extra moves
np.random.seed(42)

splits_extra = []
for i in range(20):
    shuffled = hardest_12_moves.copy()
    np.random.shuffle(shuffled)
    train = sorted(shuffled[:6])
    val = sorted(shuffled[6:])
    train_str = ' '.join([f'{p:02d}' for p in train])
    val_str = ' '.join([f'{p:02d}' for p in val])
    splits_extra.append({'train_problems': train_str, 'validation_problems': val_str})

splits_extra_df = pd.DataFrame(splits_extra)
print(f"\nCreated {len(splits_extra_df)} random 6/6 splits based on mean_extra_moves_on_success metric:")
print(splits_extra_df)

# Save
out_dir = Path("../../../tower_of_london_human_data/problem_ids/cross_validation")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "tol_problem_splits_mean_extra_moves_hardest_12_seed42.csv"
splits_extra_df.to_csv(out_path, index=False)
print(f"\nSaved splits CSV to: {out_path} (size: {out_path.stat().st_size} bytes)")


Created 20 random 6/6 splits based on mean_extra_moves_on_success metric:
       train_problems validation_problems
0   10 12 18 21 22 23   11 16 17 19 20 24
1   10 16 19 21 22 23   11 12 17 18 20 24
2   11 16 17 19 20 23   10 12 18 21 22 24
3   10 11 18 20 22 23   12 16 17 19 21 24
4   10 12 18 19 20 24   11 16 17 21 22 23
5   17 18 21 22 23 24   10 11 12 16 19 20
6   10 12 17 19 20 24   11 16 18 21 22 23
7   10 11 12 16 18 23   17 19 20 21 22 24
8   11 12 16 17 18 20   10 19 21 22 23 24
9   11 16 17 18 19 22   10 12 20 21 23 24
10  10 12 17 18 22 23   11 16 19 20 21 24
11  11 16 20 21 22 23   10 12 17 18 19 24
12  10 11 16 17 18 22   12 19 20 21 23 24
13  10 11 17 18 21 22   12 16 19 20 23 24
14  10 11 12 19 23 24   16 17 18 20 21 22
15  11 12 18 19 20 23   10 16 17 21 22 24
16  16 18 19 20 22 24   10 11 12 17 21 23
17  11 12 17 19 20 23   10 16 18 21 22 24
18  12 17 18 19 22 24   10 11 16 20 21 23
19  10 16 21 22 23 24   11 12 17 18 19 20

Saved splits CSV to: ..\..\..\tower_of_lon

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the success rate CSV files
success_rate_dir = Path("../../../tower_of_london_human_data/problems_ordered")

In [4]:
# Create 10 random 22/2 splits from 24 problems (purely random, not metric-based)
np.random.seed(42)

# Define 24 problem IDs (1..24)
probs_24 = list(range(1, 25))

splits_22_2 = []
for i in range(110):
    shuffled = probs_24.copy()
    np.random.shuffle(shuffled)
    train = sorted(shuffled[:22])
    val = sorted(shuffled[22:])
    train_str = ' '.join([f'{p:02d}' for p in train])
    val_str = ' '.join([f'{p:02d}' for p in val])
    splits_22_2.append({'train_problems': train_str, 'validation_problems': val_str})

splits_22_2_df = pd.DataFrame(splits_22_2)
print(f"\nCreated {len(splits_22_2_df)} random 22/2 splits from 24 problems:")
print(splits_22_2_df)

# Save
out_dir = Path("../../../tower_of_london_human_data/problem_ids/cross_validation")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "tol_problem_splits_random_22_2_110_v5.csv"
splits_22_2_df.to_csv(out_path, index=False)
print(f"\nSaved splits CSV to: {out_path} (size: {out_path.stat().st_size} bytes)")


Created 110 random 22/2 splits from 24 problems:
                                        train_problems validation_problems
0    01 02 03 04 05 06 08 09 10 11 12 13 14 15 16 1...               07 20
1    01 02 03 04 05 06 07 08 09 10 11 13 14 15 16 1...               12 22
2    01 02 03 04 05 06 07 08 09 10 11 12 13 15 16 1...               14 18
3    01 02 03 04 05 06 07 08 09 10 11 13 14 15 16 1...               12 18
4    01 02 03 04 05 06 07 08 09 10 11 12 14 16 17 1...               13 15
..                                                 ...                 ...
105  01 02 03 04 06 07 08 09 10 11 12 13 14 15 16 1...               05 22
106  01 03 04 05 06 07 08 09 10 11 12 13 14 15 16 1...               02 24
107  01 02 03 04 05 06 07 09 10 11 12 13 14 15 16 1...               08 21
108  01 03 04 05 06 07 08 09 10 12 13 14 15 16 17 1...               02 11
109  01 02 03 05 07 08 09 10 11 12 13 14 15 16 17 1...               04 06

[110 rows x 2 columns]

Saved splits CSV to: ..\.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the success rate CSV files
success_rate_dir = Path("../../../tower_of_london_human_data/problems_ordered")

In [12]:
# Create CSV with the 12 easiest problems by average human success rate
# (uses merged_df_sorted produced earlier in the notebook)
easy_12 = merged_df_sorted.tail(12)['TOL-ID'].str.replace('TOL-ID-', '').astype(int).tolist()
easy_12.sort()
easy_str = ', '.join([str(p) for p in easy_12])
out_dir = Path('../../../tower_of_london_human_data/problem_ids')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'easy_12_avg_success_rate.csv'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(easy_str + '\n')
print(f'Saved easiest 12 CSV to: {out_path} (size: {out_path.stat().st_size} bytes)')
print(f'Easiest 12 problems: {easy_12}')

Saved easiest 12 CSV to: ..\..\..\tower_of_london_human_data\problem_ids\easy_12_avg_success_rate.csv (size: 39 bytes)
Easiest 12 problems: [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 16]


In [13]:
# Create CSV with the 12 hardest problems by average human success rate
hard_12 = merged_df_sorted.head(12)['TOL-ID'].str.replace('TOL-ID-', '').astype(int).tolist()
hard_12.sort()
hard_str = ', '.join([str(p) for p in hard_12])
out_path = out_dir / 'hard_12_avg_success_rate.csv'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(hard_str + '\n')
print(f'Saved hardest 12 CSV to: {out_path} (size: {out_path.stat().st_size} bytes)')
print(f'Hardest 12 problems: {hard_12}')

Saved hardest 12 CSV to: ..\..\..\tower_of_london_human_data\problem_ids\hard_12_avg_success_rate.csv (size: 48 bytes)
Hardest 12 problems: [10, 11, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24]


In [8]:
# Generate 10 more 22/2 splits: 8 with 2 easy validation + 2 with 2 hard validation
# Avoid duplicates with the 110 existing splits and avoid repeated validation pairs in the new set
import pandas as pd
import numpy as np
from pathlib import Path

# Load existing 110 splits
existing_splits_path = Path("../../../tower_of_london_human_data/problem_ids/cross_validation/tol_problem_splits_random_22_2_110_v5.csv")
existing_splits_df = pd.read_csv(existing_splits_path)

# Convert existing splits to sets for efficient duplicate detection
existing_split_sets = set()
for _, row in existing_splits_df.iterrows():
    train_nums = frozenset(map(int, row['train_problems'].split()))
    val_nums = frozenset(map(int, row['validation_problems'].split()))
    existing_split_sets.add((train_nums, val_nums))

print(f"Loaded {len(existing_split_sets)} existing splits")

# Load easy and hard problem IDs
easy_12_path = Path("../../../tower_of_london_human_data/problem_ids/easy_12_avg_success_rate.csv")
hard_12_path = Path("../../../tower_of_london_human_data/problem_ids/hard_12_avg_success_rate.csv")

easy_12_text = easy_12_path.read_text(encoding='utf-8').strip()
easy_12 = sorted([int(x.strip()) for x in easy_12_text.split(',')])

hard_12_text = hard_12_path.read_text(encoding='utf-8').strip()
hard_12 = sorted([int(x.strip()) for x in hard_12_text.split(',')])

print(f"Easy 12: {easy_12}")
print(f"Hard 12: {hard_12}")

# All problems
all_problems = set(range(1, 25))

# Generate splits with constraints
np.random.seed(42)
new_splits = []
used_new_validation_pairs = set()

# 8 splits with 2 easy validation problems
while len([split for split in new_splits if split['split_type'] == 'easy']) < 8:
    # Pick 2 easy problems as validation
    val_probs = tuple(sorted(np.random.choice(easy_12, 2, replace=False).tolist()))
    val_probs_set = frozenset(val_probs)

    # Remaining 22 problems for training
    remaining = sorted(all_problems - set(val_probs))
    train_probs_set = frozenset(remaining)

    # Check for duplicate against old splits and against already generated validation pairs
    if (train_probs_set, val_probs_set) in existing_split_sets:
        continue
    if val_probs_set in used_new_validation_pairs:
        continue

    used_new_validation_pairs.add(val_probs_set)
    train_str = ' '.join([f'{p:02d}' for p in remaining])
    val_str = ' '.join([f'{p:02d}' for p in val_probs])
    new_splits.append({'train_problems': train_str, 'validation_problems': val_str, 'split_type': 'easy'})
    print(f"Generated split with easy validation: {list(val_probs)}")

# 2 splits with 2 hard validation problems
while len([split for split in new_splits if split['split_type'] == 'hard']) < 2:
    # Pick 2 hard problems as validation
    val_probs = tuple(sorted(np.random.choice(hard_12, 2, replace=False).tolist()))
    val_probs_set = frozenset(val_probs)

    # Remaining 22 problems for training
    remaining = sorted(all_problems - set(val_probs))
    train_probs_set = frozenset(remaining)

    # Check for duplicate against old splits and against already generated validation pairs
    if (train_probs_set, val_probs_set) in existing_split_sets:
        continue
    if val_probs_set in used_new_validation_pairs:
        continue

    used_new_validation_pairs.add(val_probs_set)
    train_str = ' '.join([f'{p:02d}' for p in remaining])
    val_str = ' '.join([f'{p:02d}' for p in val_probs])
    new_splits.append({'train_problems': train_str, 'validation_problems': val_str, 'split_type': 'hard'})
    print(f"Generated split with hard validation: {list(val_probs)}")

new_splits_df = pd.DataFrame(new_splits).drop(columns=['split_type'])
print(f"\nCreated {len(new_splits_df)} new constrained 22/2 splits:")
print(new_splits_df)

# Save
out_dir = Path("../../../tower_of_london_human_data/problem_ids/cross_validation")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "tol_problem_splits_random_22_2_easy_hard_validation_10.csv"
new_splits_df.to_csv(out_path, index=False)
print(f"\nSaved new constrained splits CSV to: {out_path} (size: {out_path.stat().st_size} bytes)")

Loaded 93 existing splits
Easy 12: [1, 2, 3, 4, 5, 6, 7, 8, 9, 13, 14, 16]
Hard 12: [10, 11, 12, 15, 17, 18, 19, 20, 21, 22, 23, 24]
Generated split with easy validation: [13, 14]
Generated split with easy validation: [2, 13]
Generated split with easy validation: [1, 6]
Generated split with easy validation: [9, 13]
Generated split with easy validation: [1, 3]
Generated split with easy validation: [2, 6]
Generated split with easy validation: [4, 7]
Generated split with easy validation: [6, 13]
Generated split with hard validation: [10, 21]
Generated split with hard validation: [23, 24]

Created 10 new constrained 22/2 splits:
                                      train_problems validation_problems
0  01 02 03 04 05 06 07 08 09 10 11 12 15 16 17 1...               13 14
1  01 03 04 05 06 07 08 09 10 11 12 14 15 16 17 1...               02 13
2  02 03 04 05 07 08 09 10 11 12 13 14 15 16 17 1...               01 06
3  01 02 03 04 05 06 07 08 10 11 12 14 15 16 17 1...               09 13
4 